In [5]:
import os
import re
import pandas as pd
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import xlrd

warnings.filterwarnings("ignore")

Pandas usa internamente xlrd para leer archivos xls. Los archivos de la EPH están en un formato viejo, por lo que al leerlos genera unos warnings inofensivos (uno de que no respeta el tamaño estandar de xls y otro de problemas con los encabezados). Para eliminar esos warnings definitivamente, el siguiente codigo los filtra.

In [6]:
devnull = open(os.devnull, 'w')
new_defaults = list(xlrd.open_workbook.__defaults__)
new_defaults[1] = devnull  # Reemplaza el parámetro 'logfile' por defecto de xlrd
xlrd.open_workbook.__defaults__ = tuple(new_defaults)

Cargo la CBT y genero los promedios trimestrales (hay 3 valores de cbt para cada trimestre de la EPH, lo más correcto es promediarlos).

In [7]:

data_dir = '../data/eph raw/'

# 1. Cargar y promediar la serie CBT a nivel trimestral
cbt = pd.read_csv('../data/serie_cbt.csv')
cbt['Mes_dt'] = pd.to_datetime(cbt['Mes'], format='%m/%d/%y')
cbt['ano4'] = cbt['Mes_dt'].dt.year
cbt['trimestre'] = cbt['Mes_dt'].dt.quarter
cbt_trimestral = cbt.groupby(['ano4', 'trimestre'])['CBT'].mean().reset_index()

In [8]:
cbt_trimestral.head()

,ano4,trimestre,CBT
0,2016,2,3812.366667
1,2016,3,4055.150000
2,2016,4,4232.450000
3,2017,1,4432.323333
4,2017,2,4744.123333


In [9]:
cbt.head()

,Mes,CBT,Mes_dt,ano4,trimestre
0,04/01/16,3663.66,2016-04-01,2016,2
1,05/01/16,3830.77,2016-05-01,2016,2
2,06/01/16,3942.67,2016-06-01,2016,2
3,07/01/16,4033.76,2016-07-01,2016,3
4,08/01/16,4041.87,2016-08-01,2016,3


Levantar los archivos y procesar uno por uno.

In [10]:
# 2. Identificar todos los archivos en la carpeta
# (Aseguramos que el script no falle si la carpeta aún no existe o está vacía)
if os.path.exists(data_dir):
    all_files = [f for f in os.listdir(data_dir) if f.endswith(('.xls', '.xlsx'))]
else:
    all_files = []

df_final_list = []
años = range(2016, 2027)
trimestres = range(1, 5)

función auxiliar para reconstruir unas variables que a partir de 2020/2021 empezaron a venir desagregadas. Como para este analisis esa desagregación no aporta valor, las vuelvo a convertir al formato anterior

[ TODO: definir las variables modificadas ]

In [11]:
def reconstruir_variables_base(df):
    """
    Reconstruye V2, V5, V11, V21, V22 a partir de sus subvariables _01, _02, _03
    cuando las variables base no existen o vienen vacías.
    """
    mapeo_variables = {
        'v2':  ['v2_01', 'v2_02', 'v2_03'],
        'v5':  ['v5_01', 'v5_02', 'v5_03'],
        'v11': ['v11_01', 'v11_02'],
        'v21': ['v21_01', 'v21_02', 'v21_03'],
        'v22': ['v22_01', 'v22_02', 'v22_03']
    }
    
    for var_madre, subvars in mapeo_variables.items():
        # Verificar cuáles subvariables existen en el trimestre actual
        subvars_presentes = [s for s in subvars if s in df.columns]
        
        if subvars_presentes:
            # Condición: Es 1 si CUALQUIERA de las subvariables es 1
            es_si = (df[subvars_presentes] == 1).any(axis=1)
            # Condición: Es 2 si TODAS las subvariables presentes son 2
            es_no = (df[subvars_presentes] == 2).all(axis=1)
            
            # Si var_madre no existe en el DF, la creamos
            if var_madre not in df.columns:
                df[var_madre] = 2  # default No
                
            # Asignamos los valores reconstruidos solo donde la var_madre sea nula o cuando proviene del período post-2021
            df.loc[es_si, var_madre] = 1
            df.loc[es_no & ~es_si, var_madre] = 2

    return df

In [12]:
cols_to_drop_profe = [
    "nro_hogar", "realizada", "region", "aglomerado", "pondera", "ano4", "trimestre",
    "p21", "decocur", "rdecocur", "adecocur", "pondiio", "tot_p12", "p47t", "pondii", 
    "v2_m", "v3_m", "v4_m", "v5_m", "v8_m", "v9_m", "v10_m", "v11_m", "v12_m", 
    "v18_m", "v19_am", "v21_m", "t_vi", "ipcf", "pondih", "CBT", "indigencia",
    "ix_men10", "ix_mayeq10", "pp04b_cod", "pp04d_cod", "pp11b_cod", "pp11d_cod", 
    "pp04b3_mes", "pp04b3_ano", "pp04b3_dia", "pp05b2_mes", "pp05b2_ano", "pp05b2_dia", 
    "pp06c", "pp06d", "pp08d1", "pp08d4", "pp08f1", "pp08f2", "pp08j1", "pp08j2", "pp08j3", 
    "pp09a", "pp09b", "pp09c", "pp11b2_mes", "pp11b2_ano", "pp11b2_dia", "pp11g_ano", 
    "pp11g_mes", "pp11g_dia", "h15", "imputa", "pp02c1", "pp02c2", "pp02c3", "pp02c4", 
    "pp02c5", "pp02c6", "pp02c7", "pp02c8", "vii1_1", "vii1_2", "vii2_1", "vii2_2", 
    "vii2_3", "vii2_4", "ch05"
]


años = range(2016, 2027)
trimestres = range(1, 5)

writer = None
columnas_base = None  # Guardará la estructura del primer trimestre procesado

for año in años:
    año_corto = str(año)[-2:]
    for trim in trimestres:
        re_hogar = re.compile(rf"(?i)usu_hogar_t{trim}{año_corto}\.xlsx?")
        re_indiv = re.compile(rf"(?i)usu_individual_t{trim}{año_corto}\.xlsx?")
        
        file_hogar = next((f for f in all_files if re_hogar.search(f)), None)
        file_indiv = next((f for f in all_files if re_indiv.search(f)), None)
        
        if file_hogar and file_indiv:
            df_h = pd.read_excel(
                os.path.join(data_dir, file_hogar),
                usecols=lambda x: x.lower() not in cols_to_drop_profe
            )
            df_i = pd.read_excel(
                os.path.join(data_dir, file_indiv),
                usecols=lambda x: x.lower() not in cols_to_drop_profe
            )
            
            df_h.columns = df_h.columns.str.lower()
            df_i.columns = df_i.columns.str.lower()
            
            overlap_cols = set(df_h.columns).intersection(set(df_i.columns))
            overlap_cols -= {'codusu'}
            df_h_clean = df_h.drop(columns=list(overlap_cols))
            
            merged = df_i.merge(df_h_clean, on=['codusu'], how='left')
            merged = reconstruir_variables_base(merged)
            merged = merged.reindex(columns=columnas_base)
            
            cbt_actual = cbt_trimestral[(cbt_trimestral['ano4'] == año) & (cbt_trimestral['trimestre'] == trim)]
            cbt_valor = cbt_actual['CBT'].values[0] if not cbt_actual.empty else 0
            
            merged['esPobre'] = merged['itf'] < cbt_valor
            
            # --- GARANTIZAR IDENTIDAD DE ESQUEMA ENTRE TRIMESTRES ---
            if columnas_base is None:
                columnas_base = list(merged.columns)
            else:
                # Normaliza columnas: descarta las nuevas de 2021 y crea NaNs para las ausentes
                merged = merged.reindex(columns=columnas_base)
            
            # Homogeneizar tipos de datos
            for col in columnas_base:
                if col == 'codusu':
                    merged[col] = merged[col].astype(str)
                elif col == 'esPobre':
                    merged[col] = merged[col].astype(bool)
                elif col.startswith(('ch15_cod', 'ch16_cod', 'iv', 'ii', 'pp09', 'mas_500')) or 'dec' in col:
                    merged[col] = merged[col].astype(str)
                else:
                    merged[col] = pd.to_numeric(merged[col], errors='coerce').astype('float32')

            # Escritura en Parquet
            table = pa.Table.from_pandas(merged)
            
            if writer is None:
                writer = pq.ParquetWriter('eph_consolidada_2016_2026.parquet', table.schema)
            
            writer.write_table(table)
            print(f"Procesado y guardado exitosamente: T{trim} {año}")
            
            del df_h, df_i, df_h_clean, merged, table
            gc.collect()

if writer:
    writer.close()
    print("\n¡Proceso finalizado! Base unificada y lista para el modelo.")
else:
    print("No se encontraron archivos.")

Procesado y guardado exitosamente: T2 2016
Procesado y guardado exitosamente: T3 2016
Procesado y guardado exitosamente: T4 2016
Procesado y guardado exitosamente: T1 2017
Procesado y guardado exitosamente: T2 2017
Procesado y guardado exitosamente: T3 2017
Procesado y guardado exitosamente: T4 2017
Procesado y guardado exitosamente: T1 2018
Procesado y guardado exitosamente: T2 2018
Procesado y guardado exitosamente: T3 2018
Procesado y guardado exitosamente: T4 2018
Procesado y guardado exitosamente: T1 2019
Procesado y guardado exitosamente: T2 2019
Procesado y guardado exitosamente: T3 2019
Procesado y guardado exitosamente: T4 2019
Procesado y guardado exitosamente: T1 2020
Procesado y guardado exitosamente: T2 2020
Procesado y guardado exitosamente: T3 2020
Procesado y guardado exitosamente: T4 2020
Procesado y guardado exitosamente: T1 2021
Procesado y guardado exitosamente: T2 2021
Procesado y guardado exitosamente: T3 2021
Procesado y guardado exitosamente: T4 2021
Procesado y

Ahora levanto el archivo parquet para verificar que todo esté en orden

In [13]:
# 1. Cargar el archivo Parquet consolidado
archivo_parquet = 'eph_consolidada_2016_2026.parquet'
df = pd.read_parquet(archivo_parquet)

print("--- INFORMACIÓN GENERAL DEL DATASET ---")
print(f"Total de filas (registros de individuos): {df.shape[0]:,}")
print(f"Total de columnas (variables): {df.shape[1]}")
print(f"Uso de memoria RAM: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

# 2. Distribución de la variable objetivo (esPobre)
print("--- DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (esPobre) ---")
conteo_pobreza = df['esPobre'].value_counts(dropna=False)
porcentaje_pobreza = df['esPobre'].value_counts(normalize=True, dropna=False) * 100

tabla_pobreza = pd.DataFrame({
    'Cantidad': conteo_pobreza,
    'Porcentaje (%)': porcentaje_pobreza.round(2)
})
print(tabla_pobreza)
print()

# 3. Primeras filas del DataFrame
print("--- VISTA PREVIA DE LOS DATOS ---")
print(df.head())

# 4. Chequeo de valores nulos
print("\n--- TOP 15 VARIABLES CON MÁS VALORES FALTANTES (NaN) ---")
nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df)) * 100
tabla_nulos = pd.DataFrame({
    'Cantidad Nulos': nulos,
    'Porcentaje (%)': porcentaje_nulos.round(2)
}).sort_values(by='Cantidad Nulos', ascending=False)

print(tabla_nulos.head(15))

# 5. Resumen estadístico de variables numéricas clave
print("\n--- RESUMEN ESTADÍSTICO DE VARIABLES CLAVE ---")
cols_interes = [c for c in ['itf', 'ch06', 'ix_tot', 'ch04'] if c in df.columns]
print(df[cols_interes].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

FileNotFoundError: [Errno 2] No such file or directory: 'eph_consolidada_2016_2026.parquet'

cosas que faltan:

codigo laboral: one hot encoding
ingresos no laborales 
ver en la tesis cómo hace!




medición de pobreza alternativa:
- inserción ocupacional dentro el hogar
    - Peso de no aportantes vs aportantes
    - tamaño de niños, jubilados, etc.



CBT * miembros del hogar - IFT >0

